# 🏠 Ames Housing — Model Building
### A Practical Walkthrough for IIT Undergraduates

---

## What This Session Covers

In the previous session, we cleaned, encoded, scaled and packaged everything  
into a **sklearn Pipeline** — and trained a baseline Linear Regression model.

**Result:** R² = 0.9552 | RMSE = $17,843

That number means nothing in isolation. Today we answer the real questions:

- Is that result trustworthy — or did we just get lucky with one split?
- Where does Linear Regression break down?
- Can Ridge or Lasso do better — and why?
- What happens when we use a completely different type of model — Random Forest?
- How do we fairly compare models against each other?

---

> 💡 **Dataset:** Ames Housing Dataset — 2,925 rows, 79 features, Target: `SalePrice`  
> 💡 **Tools:** scikit-learn, pandas, numpy, matplotlib, seaborn  
> 💡 **Starting point:** The preprocessing Pipeline from Lecture 1

---

## 📋 Session Roadmap

| # | Section | What We Do |
|---|---------|------------|
| 1 | **Imports & Setup** | Reload libraries and rebuild the preprocessing pipeline |
| 2 | **Linear Regression — How It Works** | The math intuition, assumptions, and where it breaks |
| 3 | **Cross Validation** | Why one train/test split lies to you — and how to fix it |
| 4 | **Ridge & Lasso** | Regularization — fixing multicollinearity and overfitting |
| 5 | **Random Forest** | A completely different way of thinking about prediction |
| 6 | **Model Comparison** | Swap models inside the same Pipeline — who wins? |
| 7 | **Summary & What Comes Next** | Key takeaways and the road ahead |

---

> 💡 **How this session works:**  
> Every model we build uses the **exact same preprocessing Pipeline** from Lecture 1.  
> The only thing that changes is the final estimator.  
> That's the power of the Pipeline pattern.

In [ ]:
# ── 1. IMPORTS & SETUP ────────────────────────────────────────────────────────

# Core
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    RobustScaler, OrdinalEncoder, OneHotEncoder
)

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

# Evaluation
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 14

print("✅ All libraries loaded successfully.")

In [ ]:
# ── Reload Data & Rebuild Preprocessing Pipeline ──────────────────────────────

# Load and clean
df = pd.read_csv("../data/AmesHousing.csv")
df = df[df["Gr Liv Area"] <= 4000].copy()
df = df.drop(columns=["Order", "PID"])

# Separate target
X = df.drop(columns=["SalePrice"])
y = np.log1p(df["SalePrice"])

# ── Column groups ──────────────────────────────────────────────────────────────
quality_cols = [
    "Exter Qual", "Exter Cond", "Bsmt Qual", "Bsmt Cond",
    "Heating QC", "Kitchen Qual", "Fireplace Qu",
    "Garage Qual", "Garage Cond", "Pool QC"
]

nominal_cols = [
    "MS Zoning", "Street", "Alley", "Lot Shape", "Land Contour",
    "Lot Config", "Land Slope", "Neighborhood", "Condition 1",
    "Condition 2", "Bldg Type", "House Style", "Roof Style",
    "Roof Matl", "Exterior 1st", "Exterior 2nd", "Mas Vnr Type",
    "Foundation", "Heating", "Central Air", "Garage Type",
    "Garage Finish", "Paved Drive", "Fence", "Misc Feature",
    "Sale Type", "Sale Condition", "Electrical", "Functional",
    "BsmtFin Type 1", "BsmtFin Type 2", "Bsmt Exposure"
]

numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# ── Transformers ───────────────────────────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler())
])

ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["None","Po","Fa","TA","Gd","Ex"]] * len(quality_cols),
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

nominal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# ── Preprocessor ───────────────────────────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("ord", ordinal_transformer, quality_cols),
    ("nom", nominal_transformer, nominal_cols)
])

# ── Train / Test Split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Data loaded and pipeline ready")
print(f"   Training set : {X_train.shape}")
print(f"   Test set     : {X_test.shape}")
print(f"   Target       : log(SalePrice) — will convert back with np.expm1()")